> ⚠️ **作業中 (Work in Progress)**:このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [ワークフロー概要](#ワークフロー概要)
- [Sequential Workflow](#sequential-workflow)
- [Group Chat Workflow](#group-chat-workflow)
- [Human-in-loop Workflow](#human-in-loop-workflow)

## 🎯 学習目標

- Microsoft Foundryワークフローのコア概念の理解
- Sequential Workflowを通じた順次タスクフローの構築
- Group Chat Workflowを通じたマルチエージェント協調の実装
- Human-in-loopパターンを通じた人間の介入ポイントの設定
- ワークフローのデプロイとプログラマティック呼び出し

## ⏱️ 予想所要時間

約20分

## 環境設定

ワークフロー 実行をための設定です.

In [ ]:
# 環境変数ロード
import json
import os
import subprocess

# PATH 環境変数の設定 (Azure CLIを見つけられるように)
possible_paths = [
  "/opt/homebrew/bin", # macOS (Apple Silicon)
  "/usr/local/bin",   # macOS (Intel) / Linux
  "/usr/bin",      # Linux / GitHub Codespaces
  "/home/linuxbrew/.linuxbrew/bin" # Linux Homebrew
]

az_path = None
try:
  result = subprocess.run(['which', 'az'], capture_output=True, text=True)
  if result.returncode == 0:
    az_path = os.path.dirname(result.stdout.strip())
except:
  pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
  paths_to_add.append(az_path)
else:
  for path in possible_paths:
    if os.path.exists(path) and path not in os.environ.get("PATH", ""):
      paths_to_add.append(path)

if paths_to_add:
  new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
  os.environ["PATH"] = new_path

# が前ノートブックで保存した設定ファイルのロード
config_file = ".foundry_config.json"
try:
  with open(config_file, 'r') as f:
    config = json.load(f)
  
  # 環境変数設定
  FOUNDRY_NAME = config.get("FOUNDRY_NAME")
  RESOURCE_GROUP = config.get("RESOURCE_GROUP")
  LOCATION = config.get("LOCATION")
  TENANT_ID = config.get("TENANT_ID")
  PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
  PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
  
  # 環境変数でも設定 (他のツールが使用できるように)
  os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
  os.environ["LOCATION"] = LOCATION
  os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
  os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
  os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
  
  print(f"✅ 設定ファイル '{config_file}'で環境変数をロードしました.")
  print(f"\n📌 Foundry Name:{FOUNDRY_NAME}")
  print(f"📌 Resource Group:{RESOURCE_GROUP}")
  print(f"📌 Location:{LOCATION}")
  print(f"📌 プロジェクトエンドポイント:{PROJECT_ENDPOINT}")
  
except FileNotFoundError:
  print(f"⚠️ '{config_file}' ファイルを見つかりません.")
  print("💡 01-setup.ipynbを先に実行して環境を設定してください.")
  raise

# 必須パッケージインストール
%pip install -q azure-ai-projects azure-identity

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用するプロジェクトエンドポイント:{PROJECT_ENDPOINT}")

## Sequential Workflow用エージェント作成

Sequential Workflowで使用するエージェントを作成します.
- **TravelPlannerAgent**:旅行目的地と日程を企画
- **LocalAgent**:現地情報を追加 (Web Search 使用)
- **TravelSummaryAgent**:最終概要およびチェックリスト作成

In [ ]:
# TravelPlannerAgent 作成
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

TRAVEL_PLANNER_INSTRUCTIONS = """あなたは旅行計画専門がです.

ロール:
1. ユーザーの旅行要件を分析します
2. 目的地の主要観光地, グルメ, 宿泊施設をおすすめします
3. 日程別旅行日程を具体的で作成します
4. 予想コストと準備物を提示します

出力形式:
- 目的地概要
- 日程別日程 (朝/昼/夜活動)
- おすすめ宿泊施設
- 予想コスト
- 準備物リスト

次のエージェントに渡す情報:全体旅行計画"""

agent_travel_planner = client.agents.create_agent(
  model="gpt-5.1",
  name="TravelPlannerAgent",
  instructions=TRAVEL_PLANNER_INSTRUCTIONS
)

print(f"✅ TravelPlannerAgent 作成完了!")
print(f"  ID:{agent_travel_planner.id}")
print(f"  Name:{agent_travel_planner.name}")

In [ ]:
# LocalAgent 作成 (Web Search も旧使用)
LOCAL_AGENT_INSTRUCTIONS = """あなたは現地情報専門がです.

ロール:
1. が前エージェントの旅行計画を受け取ります
2. Web searchを使用して最新現地情報を検索します
3. リアルタイム情報を追加します:
  - 現在天気および気候
  - 現地フェスティバルおよびがベント
  - 交通情報 (路線, 料金, 所要時間)
  - 営業時間および予約情報
  - 現地文化および注意事項

出力形式:
- 元の日程 + 現地情報補強
- 交通手段詳細情報
- 予約必要場所リスト
- 現地ヒント

次のエージェントに渡す情報:現地情報が追加された旅行計画"""

agent_local = client.agents.create_agent(
  model="gpt-5.1",
  name="LocalAgent",
  instructions=LOCAL_AGENT_INSTRUCTIONS,
  tools=[{"type":"web_search"}]
)

print(f"✅ LocalAgent 作成完了!")
print(f"  ID:{agent_local.id}")
print(f"  Tools:web_search")

In [ ]:
# TravelSummaryAgent 作成
TRAVEL_SUMMARY_INSTRUCTIONS = """あなたは旅行計画整理専門がです.

ロール:
1. が前エージェントたちの情報を総合します
2. 実行可能な最終計画で整理します
3. チェックリストを作成します

出力形式:
📋 旅行概要
- 目的地:
- 期間:
- 予算:

📅 日程概要 (一目に見るは日程)

✅ 出発前 チェックリスト
- [ ] 項目1
- [ ] 項目2

🎒 準備物チェックリスト

📞 緊急連絡先および有用な情報

最終出力:プリント可能な旅行ガイド"""

agent_travel_summary = client.agents.create_agent(
  model="gpt-5.1",
  name="TravelSummaryAgent",
  instructions=TRAVEL_SUMMARY_INSTRUCTIONS
)

print(f"✅ TravelSummaryAgent 作成完了!")
print(f"  ID:{agent_travel_summary.id}")

## Group Chat Workflow用エージェント作成

Group Chat Workflowで使用するエージェントを作成します.
- **StudentAgent**:質問に回答するは学生ロール
- **TeacherAgent**:回答を評価してフィードバックを主は教師ロール

In [ ]:
# StudentAgent 作成
STUDENT_INSTRUCTIONS = """あなたは問題に答えるはエージェントよ. 質問が来たら, 常に回答して.

ロール:
1. ユーザーの質問をが理解し回答を作成します
2. 最初の番目時もではデフォルト的な回答を提供します
3. TeacherAgentのフィードバックを受けて回答を改善します
4. すべての要件が充足される時まで回答を修正します

回答時 考慮事項:
- 日程 (日付, 時間)
- コスト (予算, が的)
- 好み (好みも, スタイル)
- 制約事項 (制限事項, 条件)

改善が必要すると TeacherAgentのフィードバックを反映して回答を補完します."""

agent_student = client.agents.create_agent(
  model="gpt-5.1",
  name="StudentAgent",
  instructions=STUDENT_INSTRUCTIONS
)

print(f"✅ StudentAgent 作成完了!")
print(f"  ID:{agent_student.id}")

In [ ]:
# TeacherAgent 作成
TEACHER_INSTRUCTIONS = """あなたは回答を評価するはエージェントよ. 回答が日程, コスト, 好みなど様々な条件にに対する考慮をしたなら [COMPLETE]がと答えて. でなければ, COMPLETEを表示しないではなく, 修正をリクエストして.

評価基準:
1. 日程:具体的な日付, 時間, 期間が含まれたはが?
2. コスト:予算, が的, コスト情報が含まれたはが?
3. 好み:ユーザーの好みも私スタイルを考慮しはが?
4. 実用性:実際で実行可能な計画のが?
5. 完成も:すべての必要な情報が含まれたはが?

レスポンス形式:
評価完了時:"[COMPLETE] すべての条件が充足されました."
改善必要時:"次の事項を補完してください:[具体的なフィードバック]"

重要:[COMPLETE]はすべての基準が充足されたを時だけ使用します."""

agent_teacher = client.agents.create_agent(
  model="gpt-5.1",
  name="TeacherAgent",
  instructions=TEACHER_INSTRUCTIONS
)

print(f"✅ TeacherAgent 作成完了!")
print(f"  ID:{agent_teacher.id}")

In [ ]:
# 作成されたエージェントリスト確認
print("=" * 80)
print("ワークフロー用エージェントリスト")
print("=" * 80)

agents = client.agents.list_agents()
workflow_agents = ["TravelPlannerAgent", "LocalAgent", "TravelSummaryAgent", "StudentAgent", "TeacherAgent"]

for agent in agents:
  if agent.name in workflow_agents:
    tools = "None"
    if agent.tools:
      tools = ", ".join([t.type if hasattr(t, 'type') else str(t) for t in agent.tools])
    print(f"\n📌 {agent.name}")
    print(f"  ID:{agent.id}")
    print(f"  Model:{agent.model}")
    print(f"  Tools:{tools}")

print("\n" + "=" * 80)
print("\n💡 が第 Azure Portalでワークフローを作成してください:")
print("  https://ai.azure.com > Build > Workflows > + Create workflow")
print("\n  Sequential Workflow:")
print("   Step 1:TravelPlannerAgent")
print("   Step 2:LocalAgent") 
print("   Step 3:TravelSummaryAgent")
print("\n  Group Chat Workflow:")
print("   Participants:StudentAgent, TeacherAgent")
print("   Termination:[COMPLETE] 含む時")

### ワークフロー 呼び出し例

In [ ]:
# Sequential Workflow 呼び出し例
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ResponseStreamEventType

WORKFLOW_NAME = "Sequential-Workflow" # ⚠️ ポータルで作成したワークフロー 名前
WORKFLOW_VERSION = "1"

# AI Project クライアント作成
project_client = AIProjectClient(
  endpoint=PROJECT_ENDPOINT,
  credential=DefaultAzureCredential(),
)

with project_client:
  workflow = {
    "name":WORKFLOW_NAME,
    "version":WORKFLOW_VERSION,
  }
  
  # OpenAI クライアントの取得
  openai_client = project_client.get_openai_client()

  # 会話作成
  conversation = openai_client.conversations.create()
  print(f"Created conversation (id:{conversation.id})")

  # ワークフロー 呼び出し (ストリーミング)
  print(f"\nCalling workflow:{WORKFLOW_NAME}...\n")
  stream = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent":{"name":workflow["name"], "type":"agent_reference"}},
    input="済州島2泊3日旅行日程作って",
    stream=True,
    metadata={"x-ms-debug-mode-enabled":"1"},
  )

  # ストリーミングがベント処理
  for event in stream:
    if event.type == ResponseStreamEventType.RESPONSE_OUTPUT_TEXT_DONE:
      print("\t", event.text)
    elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_ADDED and event.item.type == "workflow_action":
      print(f"\n{'='*60}")
      print(f"Actor - '{event.item.action_id}':")
      print(f"{'='*60}")
    elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_DONE and event.item.type == "workflow_action":
      print(f"\n✓ Workflow Item '{event.item.action_id}' is '{event.item.status}'")
      print(f" (previous item was:'{event.item.previous_action_id}')")
    elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_TEXT_DELTA:
      print(event.delta, end="", flush=True)

  # 整理
  print("\n\n✅ Workflow completed!")
  openai_client.conversations.delete(conversation_id=conversation.id)
  print("Conversation deleted")

## ワークフロー 作成

**⚠️ 重要**:ワークフローは現在 Azure Portalでだけ作成可能です.

### Azure Portalでワークフロー 作成方法:

1. [Azure AI Foundry](https://ai.azure.com) 接続
2. **Build > Workflows** メニュー
3. **+ Create workflow** クリック
4. ワークフロー タイプ選択:
  - Sequential Workflow
  - Group Chat Workflow 
  - Human-in-loop Workflow

### ワークフロー 構成例:

**Sequential Workflow (旅行計画)**
```
User Input → SearchAgent → PlannerAgent → ReviewAgent → Output
```

**Group Chat Workflow (学習議論)**
```
User Question → StudentAgent ↔ TeacherAgent → Consensus
```

作成が完了なると以下コードで実行するできるあります.

## ワークフロー 実行

ポータルで作成したワークフローを Python コードで実行します.

In [ ]:
# ワークフロー 実行コード
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ResponseStreamEventType

# クライアント作成
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# ⚠️ ワークフロー 名前とバージョン設定 (ポータルで作成したことで変更)
WORKFLOW_NAME = "Sequential-Workflow" # ⚠️ 変更必要
WORKFLOW_VERSION = "1"

with project_client:
  # OpenAI クライアントの取得
  openai_client = project_client.get_openai_client()
  
  # Conversation 作成
  conversation = openai_client.conversations.create()
  print(f"✅ Conversation 作成:{conversation.id}")
  
  # ワークフロー 実行 (ストリーミング)
  print(f"\n🚀 ワークフロー 実行中:{WORKFLOW_NAME}...\n")
  print("=" * 80)
  
  stream = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent":{"name":WORKFLOW_NAME, "type":"agent_reference"}},
    input="済州島2泊3日旅行日程作って", # ⚠️ 望むは質問で変更
    stream=True,
    metadata={"x-ms-debug-mode-enabled":"1"}
  )
  
  # ストリーミング結果処理
  for event in stream:
    if event.type == ResponseStreamEventType.RESPONSE_OUTPUT_TEXT_DELTA:
      print(event.delta, end="", flush=True)
    elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_ADDED and event.item.type == "workflow_action":
      print(f"\n\n🤖 Actor:{event.item.action_id}")
    elif event.type == ResponseStreamEventType.RESPONSE_OUTPUT_ITEM_DONE and event.item.type == "workflow_action":
      print(f"\n✅ '{event.item.action_id}' 完了 (状態:{event.item.status})")
  
  print("\n" + "=" * 80)
  print("\n✅ ワークフロー 実行完了!")
  
  # Conversation 削除
  openai_client.conversations.delete(conversation_id=conversation.id)
  print(f"🗑️ Conversation 削除になる")

### ワークフロー 設計

**ポータルで構成:**

```
TravelPlannerAgent → [ユーザー 承認] → LocalAgent → TravelSummaryAgent
```

**Approval 設定:**
- Approval message:"作成された旅行計画をレビューしてください. 承認しますか?"
- Options:Approve / Reject / Modify
- Timeout:24時間

### 💡 Human-in-loop ベスト事例

**推奨事項:**
- 承認ポイントを明確に表示
- タイムアウト設定で無限待機防止
- ユーザーにコンテキスト提供 (が前会話概要)
- 簡単な承認オプション提供 (はい/いいえ/修正)

**避けるべきすること:**
- あまりにも多いは承認ポイント
- 不明確な承認質問
- 長いタイムアウト (ユーザー 経験低下)
- 承認後 元に戻す不可能な構造

## 📚 追加リソース

- [Microsoft Foundry Workflows 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/workflow?view=foundry)
- [Microsoft Agent Framework Workflows Orchestrations パターン](https://learn.microsoft.com/en-us/agent-framework/user-guide/workflows/orchestrations/overview)

## 次のステップ

複雑なワークフローを構築しました! が第エージェントとワークフローのパフォーマンスを評価する方法を学習します:

➡️ **[06. 評価](./06-evaluations.ipynb)**:エージェントおよびワークフローの品質を体系的で評価します.